# 4. Gradient Boosting
**Use case:** Supply Chain — Predict late shipments (on-time delivery risk)

A shipper moves thousands of truckloads a month and roughly a third miss the promised delivery date. Late loads mean retailer chargebacks, stock-outs and angry customers, but expediting every load would be far too expensive. The company has carrier, lane, distance, promised transit time, pickup day, dock wait time, weather forecasts and each carrier's recent on-time record. Success means flagging the loads most likely to run late early enough to expedite, re-route or warn the customer, at a total cost lower than today.

> 💡 A dispatcher estimates delivery times. First guess: the average. Then she notices storms add delay and adjusts; then she notices long lanes add more and adjusts again. Each small correction targets what the previous estimate still got wrong - that is boosting.

Legend in code: `# 🐍 PY:` Python lesson · `# 🧠 ML:` ML lesson

In [ ]:
# Run once (Colab already has most of these)
!pip -q install numpy pandas scikit-learn scipy xgboost torch

## EASY: Five stumps learn which loads run late
Hand-code gradient boosting with five one-split trees on 12 shipments and watch the error shrink each round.

| 5W + H | |
|---|---|
| **Who** | A logistics planner who wants to understand how the late-shipment risk score is built. |
| **What** | A from-scratch boosting loop that predicts late-delivery risk from distance and a storm flag. |
| **When** | At tender time, when a load is booked with a carrier and a delivery date is promised. |
| **Where** | In a transportation management system (TMS) planning step; here on a laptop. |
| **Why** | Seeing 'fit the residuals, add a small step, repeat' by hand demystifies GradientBoosting, XGBoost and LightGBM. |
| **How** | Start from the average late rate, fit a depth-1 regression tree to the residuals, add half of its prediction, and repeat five times. |

**🧠 ML concepts:** Residuals as the target for the next tree, Decision stumps as weak learners, Learning rate (shrinkage), Additive model: sum of trees, Why classifiers boost log-odds instead of raw probabilities  
**🐍 Python concepts:** Parallel lists combined with zip, np.full to create a filled array, Appending to a list in a loop, Indexing a literal list, enumerate with tuple unpacking, sum over a generator expression

In [ ]:
# EASY: gradient boosting from scratch - each tiny tree fixes the last one's errors on late-shipment risk
import numpy as np
from sklearn.tree import DecisionTreeRegressor

# 🐍 PY: two parallel lists - distance in miles and a storm flag (1 = storm on the lane) per shipment
distance_mi = [120, 300, 450, 800, 950, 1200, 1500, 1800, 200, 650, 1100, 1700]
storm = [0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0]
# 1 = delivered late, 0 = on time
late = np.array([0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1])
# 🐍 PY: zip pairs items position by position; list(...) materialises the pairs as rows
X = np.array(list(zip(distance_mi, storm)), dtype=float)

# 🧠 ML: start with one constant guess for everyone - the overall late rate
pred = np.full(len(late), late.mean())
print(f"Starting guess for every shipment: {late.mean():.2f}\n")
learning_rate = 0.5
trees = []

for round_no in range(1, 6):
    # 🧠 ML: residual = what the current model still gets wrong (actual minus predicted)
    residual = late - pred
    # 🧠 ML: fit a tiny 'stump' (depth 1 tree) to the residuals, not to the original label
    stump = DecisionTreeRegressor(max_depth=1, random_state=42).fit(X, residual)
    # 🧠 ML: add only a fraction (learning rate) of the correction - small steps avoid overshooting
    pred = pred + learning_rate * stump.predict(X)
    trees.append(stump)
    # 🐍 PY: ['distance', 'storm'][index] picks a name by position from a literal list
    feat = ["distance", "storm"][stump.tree_.feature[0]]
    err = np.mean((late - pred) ** 2)
    print(f"round {round_no}: split on {feat:<8} at {stump.tree_.threshold[0]:>7.1f} | squared error {err:.3f}")

print("\nShipment | miles | storm | actual | predicted risk")
# 🐍 PY: enumerate gives index i; slicing [:6] shows just the first 6 rows
for i, (d, s) in enumerate(X[:6]):
    print(f"{i:>8} | {d:>5.0f} | {s:>5.0f} | {late[i]:>6} | {pred[i]:>6.2f}")

# Score a brand-new 1,400-mile shipment through a storm by summing all trees
new = np.array([[1400, 1]])
# 🐍 PY: sum() over a generator expression adds up each tree's contribution
raw = late.mean() + learning_rate * sum(t.predict(new)[0] for t in trees)
# 🧠 ML: squared-error boosting can overshoot 1.0; real classifiers boost log-odds (log-loss) instead
risk = min(max(raw, 0.0), 1.0)
print(f"\nNew 1,400 mi storm lane -> raw score {raw:.2f}, clipped risk {risk:.2f} -> "
      f"{'expedite / warn customer' if risk > 0.5 else 'standard'}")

**Try this next:**
- Set learning_rate = 1.0 and compare the error after five rounds - then try 0.1.
- Change max_depth to 2 so each tree can combine distance AND storm.
- Run 20 rounds instead of 5 and watch whether training error keeps falling (and why that can overfit).

## MEDIUM: Predict late loads across 5,000 shipments
Use scikit-learn HistGradientBoostingClassifier with native categoricals and early stopping to score late-delivery risk.

| 5W + H | |
|---|---|
| **Who** | A supply chain analytics team supporting transportation planners and customer service. |
| **What** | A late-delivery probability per shipment, checked by carrier and weather, and a 'flag the top 15%' action list. |
| **When** | When loads are tendered and again the day before pickup as weather forecasts update. |
| **Where** | In the TMS analytics layer; flagged loads appear in the planner's exception dashboard. |
| **Why** | About a third of loads miss the promise date; spotting them early lets planners expedite, re-route or warn customers. |
| **How** | Simulate 5,000 shipments, cast carrier and lane to category, train HistGradientBoosting with early stopping, then AUC, precision/recall, group checks and top-15% capture. |

**🧠 ML concepts:** Histogram-based gradient boosting, Native categorical support, Early stopping on a validation split, ROC-AUC, precision and recall, Segment-level calibration checks, Rank-based action lists  
**🐍 Python concepts:** rng.choice with probabilities, astype('category'), DataFrame.assign, groupby with named aggregation, iterrows for small tables, np.quantile for cut-offs

In [ ]:
# MEDIUM: scikit-learn HistGradientBoosting predicts which shipments will deliver late
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, precision_score, recall_score

rng = np.random.default_rng(42)
n = 5000
carriers = ["FastFreight", "BlueLine", "CheapHaul", "RegionalX"]
lanes = ["TX-OK", "CA-AZ", "IL-OH", "GA-FL", "NY-PA", "WA-OR"]
# 🐍 PY: rng.choice picks random labels from a list, with optional probabilities p
df = pd.DataFrame({
    "carrier": rng.choice(carriers, n, p=[.3, .3, .2, .2]),
    "lane": rng.choice(lanes, n),
    "distance_mi": rng.integers(50, 2200, n),
    "weather_severity": rng.integers(0, 4, n),       # 0 clear .. 3 severe storm
    "promised_days": rng.integers(1, 6, n),
    "pickup_dow": rng.integers(0, 7, n),             # 0 = Monday
    "dock_wait_hrs": rng.exponential(2, n).round(1),
})
# Hidden truth: tight promise vs distance, storms, a weak carrier, Friday pickups
# 🧠 ML: tightness = miles per promised mile-capacity; >1 means the promise is hard to meet
tightness = df.distance_mi / (df.promised_days * 500)
logit = (-4.3 + 1.6 * tightness + 0.7 * df.weather_severity + 1.1 * (df.carrier == "CheapHaul")
         + 0.5 * (df.pickup_dow == 4) + 0.15 * df.dock_wait_hrs + 0.6 * (df.lane == "IL-OH") * (df.weather_severity >= 2))
df["late"] = rng.binomial(1, 1 / (1 + np.exp(-logit)))
print(f"Shipments: {n}  late rate: {df.late.mean():.1%}")

# 🧠 ML: HistGradientBoosting can use text categories directly when typed as pandas 'category'
# 🐍 PY: .astype('category') stores repeated strings as small integer codes
for c in ["carrier", "lane"]:
    df[c] = df[c].astype("category")
X, y = df.drop(columns="late"), df.late
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

# 🧠 ML: early_stopping holds out 10% of train and stops adding trees when it stops improving
model = HistGradientBoostingClassifier(learning_rate=0.08, max_iter=400, max_leaf_nodes=15,
                                       categorical_features="from_dtype", early_stopping=True,
                                       validation_fraction=0.1, random_state=42)
model.fit(X_tr, y_tr)
print(f"Trees actually built (early stopping): {model.n_iter_}")

proba = model.predict_proba(X_te)[:, 1]
# 🧠 ML: precision = flagged loads that were late; recall = late loads we flagged
pred = (proba >= 0.5).astype(int)
print(f"ROC-AUC {roc_auc_score(y_te, proba):.3f} | precision {precision_score(y_te, pred):.2f}"
      f" | recall {recall_score(y_te, pred):.2f}")

# 🧠 ML: compare predicted vs actual late rate per carrier - a sanity check planners understand
res = X_te.assign(actual=y_te.values, predicted=proba)
# 🐍 PY: groupby + agg with named tuples produces a tidy summary table
by_carrier = res.groupby("carrier", observed=True).agg(shipments=("actual", "size"),
                                                       actual_late=("actual", "mean"),
                                                       predicted_late=("predicted", "mean"))
print("\nBy carrier:")
print(by_carrier.round(3).to_string())

by_wx = res.groupby("weather_severity").agg(actual_late=("actual", "mean"), predicted=("predicted", "mean"))
print("\nBy weather severity (0 clear .. 3 storm):")
# 🐍 PY: iterrows yields (index, row) pairs - fine for small summary tables
for sev, row in by_wx.iterrows():
    print(f"  {sev}: actual {row.actual_late:.0%} predicted {row.predicted:.0%} {'#' * int(row.predicted * 40)}")

# Business action: proactively notify customers for the riskiest 15% of shipments
# 🧠 ML: rank-based action - planners can only call so many customers, so flag the riskiest 15%
cut = np.quantile(proba, 0.85)
top = proba >= cut
print(f"\nFlag top 15% (risk >= {cut:.2f}): {y_te.values[top].mean():.0%} of them really were late "
      f"vs {y_te.mean():.0%} overall -> catches {y_te.values[top].sum() / y_te.sum():.0%} of all late loads")

**Try this next:**
- Set early_stopping=False and max_iter=400 - does test AUC improve or drop?
- Drop 'carrier' and see how much AUC falls.
- Change the flag share from 15% to 30% and compare precision vs capture.

## HARD: On-time risk engine with forward validation
Compare classic GradientBoosting with HistGradientBoosting on time-ordered data, tune both, pick an expedite threshold on dollars and monitor monthly calibration.

| 5W + H | |
|---|---|
| **Who** | Supply chain data science, transportation managers and carrier-relations teams. |
| **What** | A saved HistGradientBoosting pipeline, a cost-based expedite threshold, permutation importance and a monthly drift check. |
| **When** | Trained on Jan-Sep shipments and tested on Oct-Dec; retrained monthly in production. |
| **Where** | Batch training on the analytics platform; scores are served to the TMS for every tendered load. |
| **Why** | Late loads trigger customer chargebacks; expediting costs money too, so the cut-off must balance both, and seasonality makes time-aware validation essential. |
| **How** | Two pipelines (one-hot + GradientBoosting, ordinal + HistGradientBoosting), GridSearchCV with TimeSeriesSplit on log-loss, dummy baseline, threshold sweep on net savings, permutation importance, monthly calibration, joblib. |

**🧠 ML concepts:** Classic vs histogram gradient boosting (and the link to XGBoost/LightGBM), Time-ordered cross-validation, Log-loss as a probability-quality metric, Break-even threshold from business costs, Permutation importance, Calibration drift monitoring  
**🐍 Python concepts:** os.environ for runtime settings, Datetime columns and the .dt accessor, Functions returning different pipelines by argument, dict.items() unpacking, time.time() for timing, Tuples as running 'best so far' trackers

In [ ]:
# HARD: production on-time-delivery risk model - time-ordered CV, GBM vs HistGBM, cost-based threshold
import os
os.environ.setdefault("OMP_NUM_THREADS", "2")  # keep HistGradientBoosting's threads polite on small boxes
import tempfile
import time
import warnings
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
# 🐍 PY: module-level lists of column names keep the pipeline and the data in sync
CAT = ["carrier", "lane", "equipment"]
NUM = ["distance_mi", "promised_days", "weather_severity", "pickup_dow", "dock_wait_hrs",
       "carrier_late_rate_30d", "lane_volume_7d"]
LATE_PENALTY, EXPEDITE_COST = 800, 180   # $ chargeback per late load vs $ to expedite / reroute


def make_shipments(n=9000, seed=42):
    rng = np.random.default_rng(seed)
    df = pd.DataFrame({
        "ship_date": pd.Timestamp("2025-01-01") + pd.to_timedelta(np.sort(rng.integers(0, 365, n)), unit="D"),
        "carrier": rng.choice(["FastFreight", "BlueLine", "CheapHaul", "RegionalX"], n, p=[.3, .3, .2, .2]),
        "lane": rng.choice(["TX-OK", "CA-AZ", "IL-OH", "GA-FL", "NY-PA", "WA-OR"], n),
        "equipment": rng.choice(["dry_van", "reefer", "flatbed"], n, p=[.7, .2, .1]),
        "distance_mi": rng.integers(50, 2200, n),
        "promised_days": rng.integers(1, 6, n),
        "pickup_dow": rng.integers(0, 7, n),
        "dock_wait_hrs": rng.exponential(2, n).round(1),
        "lane_volume_7d": rng.poisson(40, n),
    })
    # 🐍 PY: the .dt accessor exposes date parts (month, weekday) of a datetime column
    month = df.ship_date.dt.month
    # 🧠 ML: seasonality - winter storms in Dec-Feb, hurricanes on GA-FL in Aug-Oct
    storm_p = 0.15 + 0.25 * month.isin([12, 1, 2]) + 0.3 * (month.isin([8, 9, 10]) & (df.lane == "GA-FL"))
    df["weather_severity"] = rng.binomial(3, storm_p)
    df["carrier_late_rate_30d"] = (0.15 + 0.12 * (df.carrier == "CheapHaul") + rng.normal(0, 0.03, n)).clip(0, 1)
    # 🧠 ML: the label is driven by interactions (reefer x storm) - a natural fit for boosted trees
    tight = df.distance_mi / (df.promised_days * 500)
    logit = (-4.2 + 1.7 * tight + 0.8 * df.weather_severity + 4.0 * df.carrier_late_rate_30d
             + 0.6 * (df.equipment == "reefer") * (df.weather_severity >= 2) + 0.5 * (df.pickup_dow == 4)
             + 0.12 * df.dock_wait_hrs + 0.01 * (df.lane_volume_7d - 40))
    df["late"] = rng.binomial(1, 1 / (1 + np.exp(-logit)))
    return df


def make_pipe(kind):
    if kind == "gbm":
        # 🧠 ML: classic GradientBoosting = exact split search, one tree per step; slower but transparent
        pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), CAT), ("num", "passthrough", NUM)])
        clf = GradientBoostingClassifier(n_estimators=150, subsample=0.8, random_state=42)
    else:
        # 🧠 ML: HistGradientBoosting bins features into 255 buckets - same idea LightGBM uses, far faster
        pre = ColumnTransformer([("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), CAT),
                                 ("num", "passthrough", NUM)])
        clf = HistGradientBoostingClassifier(max_iter=200, categorical_features=[0, 1, 2], early_stopping=True,
                                             validation_fraction=0.15, random_state=42)
    return Pipeline([("pre", pre), ("clf", clf)])


def main():
    df = make_shipments()
    # 🧠 ML: time-ordered split - train Jan-Sep, test Oct-Dec, exactly like deploying on future loads
    train, test = df[df.ship_date < "2025-10-01"], df[df.ship_date >= "2025-10-01"]
    X_tr, y_tr, X_te, y_te = train[CAT + NUM], train.late, test[CAT + NUM], test.late
    print(f"Train {len(train)} loads (late {y_tr.mean():.1%}) | Test {len(test)} loads (late {y_te.mean():.1%})")

    # 🧠 ML: TimeSeriesSplit always validates on later months than it trains on - no peeking ahead
    tscv = TimeSeriesSplit(n_splits=3)
    grids = {
        "gbm": {"clf__learning_rate": [0.05, 0.1], "clf__max_depth": [2, 3]},
        "hist": {"clf__learning_rate": [0.05, 0.1], "clf__max_leaf_nodes": [15, 31]},
    }
    # 🧠 ML: DummyClassifier predicts the average late rate for everyone - the floor to beat
    results = {"baseline": DummyClassifier(strategy="prior").fit(X_tr, y_tr)}
    # 🐍 PY: dict.items() yields (key, value) pairs we can unpack directly in the for statement
    for kind, grid in grids.items():
        t0 = time.time()
        gs = GridSearchCV(make_pipe(kind), grid, cv=tscv, scoring="neg_log_loss", n_jobs=1).fit(X_tr, y_tr)
        results[kind] = gs.best_estimator_
        print(f"{kind:<5} best {gs.best_params_} | tuned in {time.time() - t0:.1f}s")

    print("\nModel      ROC-AUC  log-loss")
    for name, m in results.items():
        p = m.predict_proba(X_te)[:, 1]
        # 🐍 PY: conditional expression - a constant model has no ranking, so AUC is 0.5 by definition
        auc = roc_auc_score(y_te, p) if name != "baseline" else 0.5
        print(f"{name:<9} {auc:>8.3f} {log_loss(y_te, p):>9.3f}")
    model = results["hist"]
    proba = model.predict_proba(X_te)[:, 1]

    # 🧠 ML: expedite when expected penalty saved exceeds expedite cost; sweep thresholds on the business $
    print("\nThreshold | expedited | late loads caught | net savings $")
    # 🐍 PY: -np.inf is a starting value any real number will beat
    best = (None, -np.inf)
    for t in np.arange(0.2, 0.81, 0.1):
        act = proba >= t
        # Assume expediting prevents 70% of late outcomes
        saved = 0.7 * LATE_PENALTY * (act & (y_te.values == 1)).sum() - EXPEDITE_COST * act.sum()
        print(f"{t:>9.1f} | {act.sum():>9} | {(act & (y_te.values == 1)).sum():>16} | {saved:>12,.0f}")
        if saved > best[1]:
            best = (round(t, 1), saved)
    print(f"Break-even risk = {EXPEDITE_COST / (0.7 * LATE_PENALTY):.2f}; best tested threshold {best[0]}")

    # 🧠 ML: permutation importance answers 'which input, if scrambled, hurts the forecast most?'
    pi = permutation_importance(model, X_te, y_te, scoring="roc_auc", n_repeats=3, random_state=42, n_jobs=1)
    print("\nPermutation importance (AUC drop):")
    for i in np.argsort(-pi.importances_mean)[:6]:
        print(f"  {X_te.columns[i]:<22} {pi.importances_mean[i]:+.4f}")

    # Monthly stability check - a light-touch drift monitor
    # 🐍 PY: .assign adds a column without modifying test; groupby month then compares means
    mon = test.assign(p=proba).groupby(test.ship_date.dt.month).agg(actual=("late", "mean"), predicted=("p", "mean"))
    print("\nMonth | actual late | predicted")
    for m, r in mon.iterrows():
        print(f"{m:>5} | {r.actual:>11.1%} | {r.predicted:>9.1%}")
    print("Governance: carrier-scorecard use must be explainable to carriers; retrain monthly, "
          "alert if calibration gap > 5 pts.")

    path = os.path.join(tempfile.mkdtemp(), "ontime_hgb.joblib")
    joblib.dump({"model": model, "threshold": best[0]}, path)
    print(f"Saved {os.path.basename(path)}")


if __name__ == "__main__":
    main()

**Try this next:**
- Change EXPEDITE_COST to 400 and see the break-even and best threshold move up.
- Replace TimeSeriesSplit with KFold(shuffle=True) and compare tuned scores - shuffling leaks seasonal patterns.
- Add 'clf__max_depth': [3, 6] to the hist grid and check whether log-loss improves.